In [ ]:
import numpy as np
import joblib as jl
import numpy as np
import pandas as pd
import tensorflow as tf
import torch

from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

#%run section_1_pre_processing.ipynb #if I must use functions defined in first notebook
print("Libraries imported successfully.")

# !pip install transformers
# !pip install tensorflow

Libraries imported successfully.


# **Section 4 – Language Models exploration**

Here applies language models like BERT to supervised intent classification by leveraging transfer learning to fine-tune pre-trained networks. It further utilizes unsupervised vector encoding to effectively harness vast amounts of data for improved model performance.

In [ ]:
# Get the GPU device name.
device_name = tf.test.gpu_device_name()

# The device name should look like the following:
if device_name == '/device:GPU:0':
    print('Found GPU at: {}'.format(device_name))
# else:
#     raise SystemError('GPU device not found')

The following code force torch to use the GPU, we need to identify and specify the GPU as the device.

In [ ]:

# If there's a GPU available...
if torch.cuda.is_available():    

    # Tell PyTorch to use the GPU.    
    device = torch.device("cuda")

    print('There are %d GPU(s) available.' % torch.cuda.device_count())

    print('We will use the GPU:', torch.cuda.get_device_name(0))

# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

NameError: name 'torch' is not defined

## **4.1 - BERT Tokenization and Data Preparation**
Take the pretrained BERT model on implementation from HuggingFace

In [ ]:
#Load Data
print("Loading data...")

# Load the DataFrame containing the text
df = jl.load('df.pkl')

# Load the target matrix 'y' created in Section 2
# y.shape should be (233035, 7)
y = jl.load('matrice_y.pkl')

# Load class names for reference 
class_names = jl.load('df_fingerprint_list.pkl')

# Extract the raw text. 
# BERT needs raw text, not TF-IDF vectors.
# column name is 'full_session' based on your previous vectorizer logic.
sentences = df['full_session'].astype(str).values

#target of our model (what must it learn)
labels = y

print(f"Loaded {len(sentences)} sentences and {len(labels)} label sets.")
print(f"Example text: {sentences[0][:100]}...")
print(f"Example label: {labels[0]}") #correct if print a sequence of seven number with value 0 or 1.

Loading data...
Loaded 233035 sentences and 233035 label sets.
Example text: enable ; system ; shell ; sh ; cat /proc/mounts ; /bin/busybox SAEMW ; cd /dev/shm ; cat .s || cp /b...
Example label: [1 1 0 0 0 0 0]


To feed our text to BERT we must split it into tokens that will be map to ehir index in the toaenizer vocabulary.

We are required to:
1. Add special tokens, CLS to the start and SEP to the end of each sentence.
2. Pad & truncate all sentences to a single constant length(max 521 token), because impact training and evaluation speed (The higher the max length, the more seconds each training epoch requires)
3. Explicitly differentiate real tokens from padding tokens with the "attention mask".



In [ ]:
# Tokenization needed for BERT
# We use the BERT tokenizer to convert text into tokens -> IDs.

print("Loading BERT tokenizer...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

# Tokenize all sentences and map the tokens to their word IDs.
input_ids = []
attention_masks = []

#to determine the max lenght 
max_len = 0

#####################
########## WE MUST FOUND THE MAX LENGHT TO WRITE IN THE FOLLOWING ENCODE_PLUS FUNCTION###########
########### APPROXIMATE VALUE TO THE NEXT PORTION OF 3 FOR EXAMPLE max_length=40 our max_lenght in function is 64

# For every sentence...
for sent in sentences:

    # Tokenize the text and add `[CLS]` and `[SEP]` tokens.
    input_ids = tokenizer.encode(sent, add_special_tokens=True)

    # Update the maximum sentence length.
    max_len = max(max_len, len(input_ids))

print('Max sentence length: ', max_len)

######################
######################


# Loop for tokenization (this might take a few minutes for 200k+ samples)
# For demonstration/speed, you might want to slice data: sentences[:10000]
# Using full dataset here:
for sent in sentences:
    encoded_dict = tokenizer.encode_plus(
                        sent,                      # Sentence to encode.
                        add_special_tokens = True, # Add '[CLS]' and '[SEP]' at 
                        max_length = 128,          # Pad & truncate all sentences.
                        padding = 'max_length', 
                        truncation = True,
                        return_attention_mask = True,   # Construct attn. masks.
                        return_tensors = 'pt',     # Return pytorch tensors.
                   )
    
    # Add the encoded sentence to the list.    
    input_ids.append(encoded_dict['input_ids'])
    
    # its attention mask (simply differentiates padding from non-padding).
    attention_masks.append(encoded_dict['attention_mask'])

# Convert the lists into tensors.
input_ids = torch.cat(input_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)
labels = torch.tensor(labels, dtype=torch.float) # Float for BCEWithLogitsLoss

print("Tokenization complete.")
print('Original: ', sentences[0])
print('Token IDs:', input_ids[0])

Loading BERT tokenizer...


NameError: name 'BertTokenizer' is not defined

Now divide up our training set to use 80% for training and 20% for validation. Then we'll also create an iterator for our dataset using the torch DataLoader class  with an iterator the entire dataset does not need to be loaded into memory.

In [ ]:
# Split data into 80% training and 20% validation
train_inputs, validation_inputs, train_labels, validation_labels = train_test_split(input_ids, labels, random_state=42, test_size=0.2)
train_masks, validation_masks, _, _ = train_test_split(attention_masks, labels, random_state=42, test_size=0.2)

# Define Batch Size (16 or 32 is recommended for BERT fine-tuning)
batch_size = 32

# Create the DataLoader for our training set.
train_dataset = TensorDataset(train_inputs, train_masks, train_labels)
train_sampler = RandomSampler(train_dataset)

#training with sampes in randomic order
train_dataloader = DataLoader(train_dataset, 
                              sampler=train_sampler, 
                              batch_size=batch_size)

# Create the DataLoader for our validation set.
validation_dataset = TensorDataset(validation_inputs, validation_masks, validation_labels)
#order it doesn't matter for validation set, so we will read them sequentially
validation_sampler = SequentialSampler(validation_dataset)
validation_dataloader = DataLoader(validation_dataset,
                                   sampler=validation_sampler, 
                                   batch_size=batch_size)

print("DataLoaders ready.")

## **4.2 - Add a last Dense Layer**
At this time we are ready to fine tune the best BERT model.
This cell defines a custom PyTorch class that uses the base BERT model as a feature extractor and adds a single Dense (Linear) layer for our 7 classes.

In [ ]:
# Define Custom BERT Architecture
import torch.nn as nn
from transformers import BertModel

class CustomBERTClassifier(nn.Module):
    def __init__(self, num_labels):
        super(CustomBERTClassifier, self).__init__()
        # 1. Load the base BERT model (pre-trained)
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        
        # 2. Add a last Dense Layer (Classifier)
        # BERT-base hidden size is 768. We map 768 -> 7 classes.
        self.classifier = nn.Linear(768, num_labels)
        
        # Optional: Add Dropout for regularization
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask):
        # Pass inputs through BERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        # Extract the 'pooler_output' (embedding of the [CLS] token)
        # This represents the entire sentence context
        pooled_output = outputs.pooler_output
        
        # Apply dropout
        pooled_output = self.dropout(pooled_output)
        
        # Pass through the Dense Layer to get logits
        logits = self.classifier(pooled_output)
        
        return logits

# Initialize the model
model = CustomBERTClassifier(num_labels=7)
model.to(device)

print("Custom BERT model initialized with a Dense Layer.")